# argentina.feriados — Pruebas interactivas

Recorrido del módulo `argentina.feriados`.

Consulta feriados argentinos desde la API pública de [argentinadatos.com](https://argentinadatos.com). Datos dinámicos — nada hardcodeado, sin scraping.

Las llamadas a la API son **diferidas**: importar `argentina` o `argentina.feriados` no pega a la red. Sólo lo hace cuando llamás a `obtener(...)`.

Requiere el extra:

```bash
pip install "argentina[feriados]"
```

## 1. Setup

In [1]:
import argentina as arg

print(f"argentina v{arg.__version__}")
print(f"API: {arg.feriados.API_URL}")

argentina v0.0.18
API: https://api.argentinadatos.com/v1/feriados/{anio}


## 2. obtener(anio)

Baja la lista completa del año. Resultado cacheado con `lru_cache(maxsize=32)` — la segunda llamada al mismo año es instantánea.

In [2]:
feriados_2026 = arg.feriados.obtener(2026)
print(f"total: {len(feriados_2026)} feriados en 2026")
feriados_2026[:5]

total: 19 feriados en 2026


[{'fecha': '2026-01-01', 'tipo': 'inamovible', 'nombre': 'Año nuevo'},
 {'fecha': '2026-02-16', 'tipo': 'inamovible', 'nombre': 'Carnaval'},
 {'fecha': '2026-02-17', 'tipo': 'inamovible', 'nombre': 'Carnaval'},
 {'fecha': '2026-03-23',
  'tipo': 'puente',
  'nombre': 'Puente turístico no laborable'},
 {'fecha': '2026-03-24',
  'tipo': 'inamovible',
  'nombre': 'Día Nacional de la Memoria por la Verdad y la Justicia'}]

In [3]:
# Cada item trae fecha (ISO), tipo y nombre
for f in feriados_2026[:10]:
    print(f"{f['fecha']}  {f['tipo']:12} {f['nombre']}")

2026-01-01  inamovible   Año nuevo
2026-02-16  inamovible   Carnaval
2026-02-17  inamovible   Carnaval
2026-03-23  puente       Puente turístico no laborable
2026-03-24  inamovible   Día Nacional de la Memoria por la Verdad y la Justicia
2026-04-02  inamovible   Día del Veterano y de los Caídos en la Guerra de Malvinas
2026-04-03  inamovible   Viernes Santo
2026-05-01  inamovible   Día del Trabajador
2026-05-25  inamovible   Día de la Revolución de Mayo
2026-06-15  trasladable  Paso a la Inmortalidad del General Martín Güemes (17/6)


In [4]:
# Tipos de feriado presentes
from collections import Counter
Counter(f['tipo'] for f in feriados_2026)

Counter({'inamovible': 12, 'trasladable': 4, 'puente': 3})

## 3. es_feriado(fecha)

Bool simple. Acepta string ISO (`YYYY-MM-DD`) o `date`.

In [5]:
from datetime import date

for v in ["2026-05-25", "2026-05-26", "2026-12-25", date(2026, 7, 9), "mala", None]:
    print(f"{v!r:20} → {arg.feriados.es_feriado(v)}")

'2026-05-25'         → True
'2026-05-26'         → False
'2026-12-25'         → True
datetime.date(2026, 7, 9) → True
'mala'               → False
None                 → False


## 4. detalle(fecha)

Devuelve el dict completo del feriado, o `None` si la fecha no es feriado.

In [6]:
arg.feriados.detalle("2026-05-25")

{'fecha': '2026-05-25',
 'tipo': 'inamovible',
 'nombre': 'Día de la Revolución de Mayo'}

In [7]:
# Día común
print(arg.feriados.detalle("2026-05-26"))
print(arg.feriados.detalle("mala"))

None
None


## 5. proximo(desde)

Próximo feriado igual o posterior a una fecha. Si no se pasa nada, usa `date.today()`. Busca en el año en curso y, si no encuentra, salta al siguiente — útil cerca de fin de año.

In [8]:
# Próximo a partir del 1 de mayo
arg.feriados.proximo("2026-05-01")

{'fecha': '2026-05-01', 'tipo': 'inamovible', 'nombre': 'Día del Trabajador'}

In [9]:
# Cerca de fin de año salta al siguiente
arg.feriados.proximo("2026-12-30")

{'fecha': '2027-01-01', 'tipo': 'inamovible', 'nombre': 'Año nuevo'}

In [10]:
# Default: hoy
arg.feriados.proximo()

{'fecha': '2026-05-25',
 'tipo': 'inamovible',
 'nombre': 'Día de la Revolución de Mayo'}

## 6. Integración con argentina.fechas

`feriados` espera fechas ISO; `fechas` te las da. Pipeline típico: parsear formato argentino → chequear si es feriado.

In [11]:
fechas_argentinas = ["25/05/2026", "26/05/2026", "09/07/2026", "25/12/2026"]

for f_arg in fechas_argentinas:
    f_iso = arg.fechas.fecha_iso(f_arg)
    es = arg.feriados.es_feriado(f_iso)
    det = arg.feriados.detalle(f_iso)
    nombre = det['nombre'] if det else "-"
    print(f"{f_arg}  iso={f_iso}  feriado={es}  {nombre}")

25/05/2026  iso=2026-05-25  feriado=True  Día de la Revolución de Mayo
26/05/2026  iso=2026-05-26  feriado=False  -
09/07/2026  iso=2026-07-09  feriado=True  Día de la Independencia
25/12/2026  iso=2026-12-25  feriado=True  Navidad


## 7. Cache

`obtener` está envuelto en `lru_cache(maxsize=32)`. Las llamadas repetidas al mismo año no pegan a la red.

In [12]:
info_antes = arg.feriados.obtener.cache_info()

# Llamadas repetidas
for _ in range(5):
    arg.feriados.obtener(2026)

info_despues = arg.feriados.obtener.cache_info()
print("antes: ", info_antes)
print("despues:", info_despues)
print(f"hits nuevos: {info_despues.hits - info_antes.hits}")

antes:  CacheInfo(hits=19, misses=2, maxsize=32, currsize=2)
despues: CacheInfo(hits=24, misses=2, maxsize=32, currsize=2)
hits nuevos: 5


In [13]:
# Si querés forzar refresh:
# arg.feriados.obtener.cache_clear()
arg.feriados.obtener.cache_info()

CacheInfo(hits=24, misses=2, maxsize=32, currsize=2)

## 8. Tests automáticos

Los tests usan `monkeypatch` para mockear `requests.get` y correr sin red.

```bash
cd /Users/tobiasyatche/argentina
pytest tests/test_feriados.py -v
```

## Notas sueltas / TODOs

- API usada: [argentinadatos.com](https://argentinadatos.com) — endpoint `/v1/feriados/{anio}`. Pública, sin auth.
- `requests` es opcional. Importar `argentina.feriados` no requiere tenerlo instalado; si llamás a una función sin tenerlo, levantás `ImportError` con la instrucción de instalación.
- Cache vive en memoria (`lru_cache`). No persiste entre runs — si querés cache en disco, agregar `joblib.Memory` o similar (fuera de scope por ahora).
- Feriados puente / fines de semana largos / días no laborables provinciales: lo que devuelva la API. No agrego lógica propia.
- Para fechas en formato argentino (`dd/mm/yyyy`), normalizar primero con `arg.fechas.fecha_iso(...)` y después pasar a `feriados`.